In [ ]:
import os, io, random, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU non actif -> Settings > Accelerator > GPU"
print("device:", device, torch.cuda.get_device_name(0))


In [ ]:
BASE = Path("/kaggle/input")
comp = next(p.parent for p in BASE.rglob("sample_submission.csv"))

DET_BACKBONE = "tf_efficientnet_b0_ns"
SEG_BACKBONE = "tf_efficientnetv2_s"
DET_IMG = 320
SEG_IMG = 512
DET_BATCH = 24
SEG_BATCH = 8
DET_EPOCHS = 7
SEG_EPOCHS = 22
LR = 4e-4
PRETRAINED = True
NFOLD = 5
MS_SCALES = (1.0, 0.875)

def load_seg(path):
    df = pd.read_csv(path, dtype=str).fillna("")
    df.columns = ["image_id", "polygon"]
    return df

train_df = load_seg(comp / "train" / "segmentation.csv")
val_df = load_seg(comp / "val" / "segmentation.csv")
sub = pd.read_csv(comp / "sample_submission.csv")
test_imgs = comp / "test" / "images"

def parse_polys(s):
    out = []
    for part in s.split(";"):
        v = part.split()
        if len(v) >= 6 and len(v) % 2 == 0:
            out.append(np.array(v, np.float32).reshape(-1, 2))
    return out

def has_poly(s):
    return len(parse_polys(s)) > 0

all_df = pd.concat([train_df.assign(src="train"), val_df.assign(src="val")], ignore_index=True)
all_df["cat2"] = all_df.polygon.map(has_poly).astype(int)
path_of = {r.image_id: comp / r.src / "images" / r.image_id for _, r in all_df.iterrows()}
poly_of = dict(zip(all_df.image_id, all_df.polygon))
cat2_lbl = dict(zip(all_df.image_id, all_df.cat2))
print("total:", len(all_df), "| cat2:", all_df.cat2.sum())


In [ ]:
def ela(pil, q=90, scale=12):
    buf = io.BytesIO(); pil.save(buf, "JPEG", quality=q); buf.seek(0)
    re = Image.open(buf).convert("RGB")
    d = np.abs(np.asarray(pil, np.int16) - np.asarray(re, np.int16))
    return np.clip(d * scale, 0, 255).astype(np.uint8)

def noise(pil, scale=4):
    a = np.asarray(pil, np.float32)
    res = a - cv2.GaussianBlur(a, (0, 0), 1.0)
    return np.clip(res * scale + 128, 0, 255).astype(np.uint8)

forensic6 = lambda pil: np.concatenate([ela(pil), noise(pil)], axis=2)
rep9 = lambda pil: np.concatenate([np.asarray(pil), ela(pil), noise(pil)], axis=2)

def poly_to_mask(poly_str, size):
    m = np.zeros((size, size), np.uint8)
    polys = parse_polys(poly_str)
    if polys:
        cv2.fillPoly(m, [(p * size).round().astype(np.int32) for p in polys], 1)
    return m

def build_cache(ids, fn, size):
    def make(name):
        p = path_of[name] if name in path_of else test_imgs / name
        with Image.open(p) as im:
            im = im.convert("RGB").resize((size, size), Image.BILINEAR)
            return name, fn(im)
    with ThreadPoolExecutor(max_workers=8) as ex:
        return dict(ex.map(make, list(ids)))


class ClsSet(Dataset):
    def __init__(self, ids, cache, tf, labels=None):
        self.ids = list(ids); self.cache = cache; self.tf = tf; self.labels = labels
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        n = self.ids[i]; x = self.tf(image=self.cache[n])["image"]
        return (x, n) if self.labels is None else (x, torch.tensor(self.labels[n], dtype=torch.float32))


class SegSet(Dataset):
    def __init__(self, ids, cache, tf, masks=None):
        self.ids = list(ids); self.cache = cache; self.tf = tf; self.masks = masks
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        n = self.ids[i]
        if self.masks is None:
            return self.tf(image=self.cache[n])["image"], n
        t = self.tf(image=self.cache[n], mask=self.masks[n])
        return t["image"], t["mask"].unsqueeze(0).float()


def cls_tf(size, train):
    m, s = [0.5]*6, [0.5]*6
    ops = [A.PadIfNeeded(size, size, border_mode=0)]
    if train: ops += [A.HorizontalFlip(0.5), A.VerticalFlip(0.5)]
    ops += [A.Normalize(m, s, max_pixel_value=255.0), ToTensorV2()]
    return A.Compose(ops)

def seg_tf(train):
    m, s = [0.5]*9, [0.5]*9
    ops = [A.PadIfNeeded(SEG_IMG, SEG_IMG, border_mode=0)]
    if train:
        ops += [A.HorizontalFlip(0.5), A.VerticalFlip(0.5),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5)]
    ops += [A.Normalize(m, s, max_pixel_value=255.0), ToTensorV2()]
    return A.Compose(ops)

dl_args = dict(num_workers=2, pin_memory=True)


In [ ]:
import copy

class Dec(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c + skip_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.conv(torch.cat([x, skip], 1))

class UNet(nn.Module):
    def __init__(self, backbone, in_chans):
        super().__init__()
        self.enc = timm.create_model(backbone, features_only=True, pretrained=PRETRAINED, in_chans=in_chans)
        c = self.enc.feature_info.channels()
        self.d4 = Dec(c[4], c[3], 256); self.d3 = Dec(256, c[2], 128)
        self.d2 = Dec(128, c[1], 64); self.d1 = Dec(64, c[0], 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        f1, f2, f3, f4, f5 = self.enc(x)
        d = self.d4(f5, f4); d = self.d3(d, f3); d = self.d2(d, f2); d = self.d1(d, f1)
        d = F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(d)

bce = nn.BCEWithLogitsLoss()
def dice_loss(logit, tgt, eps=1.0):
    p = torch.sigmoid(logit)
    num = 2*(p*tgt).sum((2, 3)) + eps
    den = p.sum((2, 3)) + tgt.sum((2, 3)) + eps
    return (1 - num/den).mean()

@torch.no_grad()
def cls_predict(model, dl, tta=False):
    model.eval(); out = []
    for x, _ in dl:
        x = x.to(device)
        views = [x, torch.flip(x, [3]), torch.flip(x, [2])] if tta else [x]
        with torch.amp.autocast("cuda"):
            p = sum(torch.sigmoid(model(v).squeeze(1)) for v in views) / len(views)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def _one(model, v):
    with torch.amp.autocast("cuda"):
        return torch.sigmoid(model(v))

@torch.no_grad()
def seg_predict(model, dl, ms=False):

    model.eval(); out = []
    scales = MS_SCALES if ms else (1.0,)
    for x, _ in dl:
        x = x.to(device)
        acc = 0.0
        for sc in scales:
            xi = x if sc == 1.0 else F.interpolate(x, scale_factor=sc, mode="bilinear", align_corners=False)
            p = _one(model, xi)
            p = p + torch.flip(_one(model, torch.flip(xi, [3])), [3])
            p = p + torch.flip(_one(model, torch.flip(xi, [2])), [2])
            p = p / 3
            if sc != 1.0:
                p = F.interpolate(p, size=x.shape[-2:], mode="bilinear", align_corners=False)
            acc = acc + p
        out.append((acc / len(scales)).squeeze(1).float().cpu().numpy())
    return np.concatenate(out)


In [ ]:
def clean_mask(binm, min_area=40):
    m = binm.astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    out = np.zeros_like(m)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            out[lab == i] = 1
    return out.astype(bool)

def mask_to_poly(binmask):
    cnts, _ = cv2.findContours(binmask.astype(np.uint8), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    parts = []
    for c in cnts:
        if cv2.contourArea(c) < 12: continue
        eps = 0.004 * cv2.arcLength(c, True)
        ap = cv2.approxPolyDP(c, eps, True).reshape(-1, 2).astype(np.float32) / binmask.shape[0]
        if len(ap) >= 3:
            parts.append(" ".join(f"{x:.6f} {y:.6f}" for x, y in ap))
    return ";".join(parts) if parts else " "

def pred_to_poly(prob, pt):
    return mask_to_poly(clean_mask(prob > pt))

def iou(g, p):
    u = (g | p).sum()
    return 1.0 if u == 0 else (g & p).sum() / u

PIX_GRID = np.linspace(0.35, 0.60, 6)


In [ ]:
det_cache = build_cache(all_df.image_id, forensic6, DET_IMG)
det_test_cache = build_cache(sub.image_id, forensic6, DET_IMG)
ids = all_df.image_id.values
y = all_df.cat2.values
det_oof = np.zeros(len(all_df)); det_test_prob = np.zeros(len(sub))

for k, (tr, va) in enumerate(StratifiedKFold(NFOLD, shuffle=True, random_state=SEED).split(ids, y)):
    tr_dl = DataLoader(ClsSet(ids[tr], det_cache, cls_tf(DET_IMG, True), cat2_lbl), DET_BATCH, shuffle=True, drop_last=True, **dl_args)
    va_dl = DataLoader(ClsSet(ids[va], det_cache, cls_tf(DET_IMG, False)), DET_BATCH, shuffle=False, **dl_args)
    m = timm.create_model(DET_BACKBONE, pretrained=PRETRAINED, num_classes=1, in_chans=6).to(device)
    pw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
    lf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    scd = torch.optim.lr_scheduler.OneCycleLR(opt, LR, epochs=DET_EPOCHS, steps_per_epoch=len(tr_dl))
    scaler = torch.amp.GradScaler("cuda")
    best, bs = 0.0, None
    for e in range(DET_EPOCHS):
        m.train()
        for xx, yy in tr_dl:
            xx, yy = xx.to(device), yy.to(device); opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = lf(m(xx).squeeze(1), yy)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); scd.step()
        f1 = f1_score(y[va], (cls_predict(m, va_dl) > 0.5).astype(int))
        if f1 > best: best, bs = f1, copy.deepcopy(m.state_dict())
    m.load_state_dict(bs)
    det_oof[va] = cls_predict(m, va_dl, tta=True)
    det_test_prob += cls_predict(m, DataLoader(ClsSet(sub.image_id, det_test_cache, cls_tf(DET_IMG, False)), DET_BATCH, **dl_args), tta=True) / NFOLD
    print(f"[det] fold {k}  f1={best:.4f}")
    del m; gc.collect(); torch.cuda.empty_cache()
print("detecteur OOF f1:", f1_score(y, (det_oof > 0.5).astype(int)))
del det_cache, det_test_cache; gc.collect(); torch.cuda.empty_cache()


In [ ]:
cat2_ids = all_df[all_df.cat2 == 1].image_id.values
seg_cache = build_cache(cat2_ids, rep9, SEG_IMG)
seg_mask = {n: poly_to_mask(poly_of[n], SEG_IMG) for n in cat2_ids}

iou_by_pix = {}

for k, (tr, va) in enumerate(KFold(NFOLD, shuffle=True, random_state=SEED).split(cat2_ids)):
    tr_ids, va_ids = cat2_ids[tr], cat2_ids[va]
    gt_va = np.stack([seg_mask[n] for n in va_ids]).astype(bool).reshape(len(va_ids), -1)
    tr_dl = DataLoader(SegSet(tr_ids, seg_cache, seg_tf(True), seg_mask), SEG_BATCH, shuffle=True, drop_last=True, **dl_args)
    va_dl = DataLoader(SegSet(va_ids, seg_cache, seg_tf(False)), SEG_BATCH, shuffle=False, **dl_args)
    seg = UNet(SEG_BACKBONE, 9).to(device)
    opt = torch.optim.AdamW(seg.parameters(), lr=LR, weight_decay=1e-4)
    scd = torch.optim.lr_scheduler.OneCycleLR(opt, LR, epochs=SEG_EPOCHS, steps_per_epoch=len(tr_dl))
    scaler = torch.amp.GradScaler("cuda")
    best, bs = 0.0, None
    for e in range(SEG_EPOCHS):
        seg.train()
        for xx, mm in tr_dl:
            xx, mm = xx.to(device), mm.to(device); opt.zero_grad()
            with torch.amp.autocast("cuda"):
                lo = seg(xx); loss = bce(lo, mm) + dice_loss(lo, mm)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); scd.step()
        pv = seg_predict(seg, va_dl).reshape(len(va_ids), -1) > 0.5
        inter, union = (gt_va & pv).sum(1), (gt_va | pv).sum(1)
        iou_e = np.where(union == 0, 1.0, inter/np.maximum(union, 1)).mean()
        if iou_e > best: best, bs = iou_e, copy.deepcopy(seg.state_dict())
    seg.load_state_dict(bs)


    for n, pm in zip(va_ids, seg_predict(seg, va_dl, ms=True)):
        g = seg_mask[n].astype(bool)
        iou_by_pix[n] = [iou(g, poly_to_mask(pred_to_poly(pm, pt), SEG_IMG).astype(bool)) for pt in PIX_GRID]

    torch.save(seg.state_dict(), f"/kaggle/working/seg_fold{k}.pt")
    print(f"[seg] fold {k}  best_iou_cat2={best:.4f}")
    del seg, tr_dl, va_dl, gt_va; gc.collect(); torch.cuda.empty_cache()

del seg_cache, seg_mask; gc.collect(); torch.cuda.empty_cache()
print("seg CV termine")


In [ ]:
seg_test_cache = build_cache(sub.image_id, rep9, SEG_IMG)
te_dl = DataLoader(SegSet(sub.image_id, seg_test_cache, seg_tf(False)), SEG_BATCH, **dl_args)
seg_test = np.zeros((len(sub), SEG_IMG, SEG_IMG), np.float32)

for k in range(NFOLD):
    seg = UNet(SEG_BACKBONE, 9).to(device)
    seg.load_state_dict(torch.load(f"/kaggle/working/seg_fold{k}.pt"))
    seg_test += seg_predict(seg, te_dl, ms=True) / NFOLD
    del seg; gc.collect(); torch.cuda.empty_cache()

del seg_test_cache, te_dl; gc.collect()
print("predictions test pretes")


In [ ]:
idx = {n: i for i, n in enumerate(all_df.image_id)}
is_c2 = all_df.set_index("image_id").cat2.to_dict()

best = (0.0, 0.5, 0.45)
for dt in np.linspace(0.3, 0.7, 9):
    for j, pt in enumerate(PIX_GRID):
        tot = 0.0
        for n in all_df.image_id:
            if det_oof[idx[n]] < dt:
                tot += 1.0 if is_c2[n] == 0 else 0.0
            else:
                tot += iou_by_pix[n][j] if is_c2[n] == 1 else 0.0
        m = tot / len(all_df)
        if m > best[0]: best = (m, dt, pt)
BEST_IOU, DET_T, PIX_T = best
print(f"OOF IoU={BEST_IOU:.4f}  seuil_det={DET_T:.3f}  seuil_pixel={PIX_T:.3f}")

rows = [" " if det_test_prob[i] < DET_T else pred_to_poly(seg_test[i], PIX_T) for i in range(len(sub))]
out = sub[["image_id"]].copy(); out["polygon"] = rows
assert out.image_id.is_unique and len(out) == len(sub)
out.to_csv("submission.csv", index=False)
print("vides:", sum(r == " " for r in rows), "| polygones:", sum(r != " " for r in rows))
